# Opdracht 1: Voorspel herseninfarct (MedPredict)

**Team:** *AMK-KLAS B*

**Teamleden en Kaggle-gebruikersnamen:**

| Naam | Kaggle-gebruikersnaam |
|---|---|
| **Ahmed Bashabeeb 25097172** | ahmedbashabeeb |
| **Khaleel Al-Aqel 24053872** | khaleelalaqel| 
| **Mehmet Kilinc 25135007** | ** |



## 1. Inleiding

In deze opdracht ontwikkelen we voor MedPredict een machine learning-model dat voorspelt of een patiënt een herseninfarct (stroke) krijgt, op basis van gegevens. We werken hiervoor met een dataset van ruim 33.000 patiënten en leveren onze voorspellingen in via een Kaggle-competitie. Naast een zo goed mogelijk voorspellend model kijken we ook naar de medische context: een gemiste


## 2. Data

We gebruiken drie bestanden uit de Kaggle-competitie
["Predicting stroke 2026-27"](https://www.kaggle.com/competitions/predicting-stroke-2026-27):

- **train.csv** — 33.550 patiënten, inclusief doelvariabele `stroke` (0/1).
- **test.csv** — 8.388 patiënten, zonder `stroke`-kolom.
- **submission_example.csv** — voorbeeldformaat voor de Kaggle-inlevering (`id`, `stroke`).


In [7]:
# Gedeelde imports (door iedereen te gebruiken/aan te vullen)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (6, 4)

RANDOM_STATE = 42 


**pd.set_option** zorgt ervoor dat pandas alle kolommen toont wanneer je een dataframe print, in plaats van kolommen af te korten met ...
<br>

**plt.rcParams** stelt de standaardgrootte in van elke grafiek die we met matplotlib maken

<br>

### Data inladen

In [8]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

## 3. Opdracht 1: Exploratieve Data Analyse (EDA)


### 3.1 Kolombeschrijving

| Kolom | Betekenis | Type |
|---|---|---|
| id | Uniek patiënt-identificatienummer | identifier |
| gender | Geslacht van de patiënt: "Male", "Female" of "Other" | categorisch |
| age | Leeftijd van de patiënt in jaren | numeriek |
| hypertension | 0 = geen hoge bloeddruk, 1 = wel hoge bloeddruk | categorisch (binair) |
| heart_disease | 0 = geen hartaandoening, 1 = wel een hartaandoening | categorisch (binair) |
| ever_married | Burgerlijke staat: "No" of "Yes" | categorisch |
| work_type | Type werk: "children", "Govt_job", "Never_worked", "Private", "Self-employed" | categorisch |
| Residence_type | Woonomgeving: "Rural" of "Urban" | categorisch |
| avg_glucose_level | Gemiddeld glucosegehalte in het bloed | numeriek |
| bmi | Body mass index | numeriek |
| smoking_status | Rookgedrag: "formerly smoked", "never smoked", "smokes", "Unknown" | categorisch |
| stroke | **Doelvariabele**: 1 = patiënt heeft een herseninfarct gehad, 0 = niet | categorisch (target) |




------------

## 4. Opdracht 2: Evaluatie

### 4.1 Evaluatie-klasse

### 4.2 Metrics voor classificatie

### 4.3 Metrics toegepast op een model

### 4.4 Waarom de F1-score?

In [9]:
# Imports voor evaluatie en modellen
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (ConfusionMatrixDisplay, confusion_matrix,
                             make_scorer, precision_score)
from imblearn.pipeline import Pipeline

### 4.1 Evaluatie-klasse

In [10]:
# Tijdelijk: totdat de Preprocessing-klasse van Ahmed klaar is
X = train_df.drop(columns=["id", "stroke"]).astype(float)
y = train_df["stroke"]

In [11]:
class ModelEvaluator:
    """Evalueert modellen met dezelfde folds en metrics voor het team."""

    # zero_division=0: geen waarschuwing als een model nooit 1 voorspelt
    SCORING = {
        "f1": "f1",
        "precision": make_scorer(precision_score, zero_division=0),
        "recall": "recall",
        "accuracy": "accuracy",
        "roc_auc": "roc_auc",
    }

    def __init__(self, n_splits=5, random_state=RANDOM_STATE):
        self.cv = StratifiedKFold(n_splits=n_splits, shuffle=True,
                                  random_state=random_state)
        self.results = []

    @staticmethod
    def build_pipeline(model, sampler=None, scale=True):
        """Bouw een pipeline: (scaler) -> (sampler) -> model."""
        steps = []
        if scale:
            steps.append(("scaler", StandardScaler()))
        if sampler is not None:
            steps.append(("sampler", sampler))
        steps.append(("model", model))
        return Pipeline(steps)

    def evaluate(self, name, pipeline, X, y):
        """Cross-validatie met alle metrics; resultaat wordt opgeslagen."""
        scores = cross_validate(pipeline, X, y, cv=self.cv,
                                scoring=self.SCORING, n_jobs=-1)
        row = {"model": name}
        for metric in self.SCORING:
            row[metric] = scores[f"test_{metric}"].mean()
        row["f1_std"] = scores["test_f1"].std()
        self.results.append(row)
        return row

    def results_table(self):
        """Scoretabel van alle geëvalueerde modellen, gesorteerd op f1."""
        return (pd.DataFrame(self.results)
                .sort_values("f1", ascending=False)
                .reset_index(drop=True)
                .round(4))

### 4.3 Metrics toegepast op een model

In [ ]:
# Prompt 1: Evaluatie-klasse
evaluator = ModelEvaluator()

lr_baseline = evaluator.build_pipeline(
    LogisticRegression(class_weight="balanced", max_iter=1000,
                       random_state=RANDOM_STATE))

evaluator.evaluate("LR baseline (balanced)", lr_baseline, X, y)
evaluator.results_table()

,model,f1,precision,recall,accuracy,roc_auc,f1_std
0,LR baseline (balanced),0.0907,0.048,0.8143,0.7485,0.8573,0.0051


Exception ignored in: <function ResourceTracker.__del__ at 0x107f61ee0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x10401dee0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x10738dee0>
Traceback (most recent call last